# **Classification: AdaBoost Classifier**

## **Justification of Preprocessing Strategy**

### **Scale Invariance**
AdaBoost (Adaptive Boosting) natively utilizes shallow Decision Trees—often called "Decision Stumps" (`max_depth=1`)—as its base estimators. Because these stumps partition data based on single feature thresholds rather than calculating spatial distance metrics, AdaBoost inherits complete scale invariance. Standardization or Normalization will not alter the model's decision boundaries. To maintain computational efficiency and analytical clarity, we will train the model using the **Original, Unscaled Data**.

### **Why We Do Not Use the Decision Tree Champion**
Unlike our Bagging or Random Forest experiments, we **do not** inject our optimized Decision Tree champion (`max_depth=10`) into AdaBoost. AdaBoost relies on a sequential *Boosting* mechanism: it requires "Weak Learners" that perform just slightly better than random guessing. If we provided deep, complex trees, the first iteration would severely overfit and achieve near-zero error, leaving subsequent trees with no mistakes to correct, completely breaking the adaptive learning loop. We rely exclusively on the default stumps.

### **Adaptive Learning Mechanism**
The core strength of AdaBoost lies in its sequential weighting. It assigns weights to each training sample. In every iteration, it amplifies the weights of the patients that were incorrectly classified by the previous stump. This forces the next weak learner to hyper-focus on the most difficult clinical cases (e.g., boundary diabetic patients), gradually building a highly accurate ensemble.


## **Experiment Design**

We defined a tournament of 3 optimization levels to find the perfect balance between model complexity and learning speed. We strictly log **both Train and Test metrics** across all runs to ensure the boosting process stops before it starts memorizing noise:

* **Baseline**: Strict defaults (`n_estimators=50`, `learning_rate=1.0`) as per Scikit-Learn documentation to establish a performance floor.
* **GridSearchCV**: A systematic 3-fold cross-validated search over the ensemble size (`n_estimators`) and the `learning_rate` to identify the optimal boosting configuration.
* **Optuna Optimization**: Bayesian optimization to explore the continuous logarithmic space of the `learning_rate` alongside `n_estimators`, targeting maximum generalizable Recall.

In [1]:
import pandas as pd
import numpy as np
import time
import mlflow
import optuna
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.ensemble import AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import recall_score, accuracy_score, f1_score

# MLflow Configuration
mlflow.set_tracking_uri("sqlite:///C:/Users/Tiago Silva/Uni/OneDrive - Universidade Portucalense/Ambiente de Trabalho/Uni/3ano2sem/LAD/Grupo5_ProjetoLAD_Parte2/TrabalhoLAD/models/mlflow.db")
mlflow.set_experiment("Classification_AdaBoost")

# Data Loading and Preparation
df = pd.read_csv("C:\\Users\\Tiago Silva\\Uni\\OneDrive - Universidade Portucalense\\Ambiente de Trabalho\\Uni\\3ano2sem\\LAD\\Grupo5_ProjetoLAD_Parte2\\TrabalhoLAD\\data\\diabetes_dataset_new_variables.csv")

categorical_cols = [
    'gender', 'ethnicity', 'smoking_status', 'education_level',
    'employment_status', 'age_groups', 'weight_status', 'income_level'
]

# One-Hot Encoding
df_final = pd.get_dummies(df, columns=categorical_cols, drop_first=True)

X = df_final.drop(["diagnosed_diabetes", "diabetes_stage"], axis=1)
y = df_final['diagnosed_diabetes']

# Split data (80/20) maintaining class proportion
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

def log_classification_metrics(model, X_tr, y_tr, X_te, y_te, duration):
    """Logs both Train and Test metrics to explicitly monitor the Overfitting Gap"""
    y_tr_pred = model.predict(X_tr)
    y_te_pred = model.predict(X_te)
    
    # Train Partition Metrics
    mlflow.log_metric("recall_train", recall_score(y_tr, y_tr_pred))
    mlflow.log_metric("accuracy_train", accuracy_score(y_tr, y_tr_pred))
    mlflow.log_metric("f1_train", f1_score(y_tr, y_tr_pred))
    
    # Test Partition Metrics
    mlflow.log_metric("recall_test", recall_score(y_te, y_te_pred))
    mlflow.log_metric("accuracy_test", accuracy_score(y_te, y_te_pred))
    mlflow.log_metric("f1_test", f1_score(y_te, y_te_pred))
    
    mlflow.log_metric("fit_time", duration)

# ---------------------------------------------------------
# RUN 1: BASELINE 
# ---------------------------------------------------------
with mlflow.start_run(run_name="AdaBoost_Baseline"):
    ada_base = AdaBoostClassifier(random_state=42)
    
    start_time = time.time()
    ada_base.fit(X_train, y_train)
    duration = time.time() - start_time
    
    mlflow.log_params(ada_base.get_params())
    mlflow.log_param("optimization", "none_default")
    
    log_classification_metrics(ada_base, X_train, y_train, X_test, y_test, duration)

# ---------------------------------------------------------
# RUN 2: GRIDSEARCHCV 
# ---------------------------------------------------------
with mlflow.start_run(run_name="AdaBoost_GridSearch"):
    param_grid = {
        'n_estimators': [50, 100, 200],
        'learning_rate': [0.01, 0.1, 1.0]
    }
    
    grid = GridSearchCV(
        AdaBoostClassifier(random_state=42),
        param_grid, cv=3, scoring='recall', n_jobs=-1
    )
    
    start_time = time.time()
    grid.fit(X_train, y_train)
    duration = time.time() - start_time
    
    best_ada_grid = grid.best_estimator_
    
    mlflow.log_params(grid.best_params_)
    mlflow.log_param("optimization", "GridSearchCV")
    
    log_classification_metrics(best_ada_grid, X_train, y_train, X_test, y_test, duration)

# ---------------------------------------------------------
# RUN 3: OPTUNA 
# ---------------------------------------------------------
def objective(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 10, 300),
        "learning_rate": trial.suggest_float("learning_rate", 0.001, 1.0, log=True)
    }
    
    model = AdaBoostClassifier(random_state=42, **params)
    
    # Using 3-fold CV entirely on Train data to prevent leakage
    score = cross_val_score(model, X_train, y_train, cv=3, scoring='recall', n_jobs=-1).mean()
    return score

with mlflow.start_run(run_name="AdaBoost_Optuna"):
    study = optuna.create_study(direction="maximize")
    start_time = time.time()
    study.optimize(objective, n_trials=15) 
    duration = time.time() - start_time
    
    # Train final champion model
    best_ada_opt = AdaBoostClassifier(random_state=42, **study.best_params)
    best_ada_opt.fit(X_train, y_train)
    
    mlflow.log_params(study.best_params)
    mlflow.log_param("optimization", "optuna")
    
    log_classification_metrics(best_ada_opt, X_train, y_train, X_test, y_test, duration)

[I 2026-05-20 23:02:30,989] A new study created in memory with name: no-name-7fcb8bbb-7551-4aef-853d-5377e52132ae
[I 2026-05-20 23:02:32,444] Trial 0 finished with value: 0.8520979813842532 and parameters: {'n_estimators': 12, 'learning_rate': 0.9489585194956307}. Best is trial 0 with value: 0.8520979813842532.
[I 2026-05-20 23:02:45,843] Trial 1 finished with value: 0.8520979813842532 and parameters: {'n_estimators': 131, 'learning_rate': 0.004395586644913904}. Best is trial 0 with value: 0.8520979813842532.
[I 2026-05-20 23:03:01,688] Trial 2 finished with value: 0.8520979813842532 and parameters: {'n_estimators': 151, 'learning_rate': 0.09949923033903044}. Best is trial 0 with value: 0.8520979813842532.
[I 2026-05-20 23:03:11,501] Trial 3 finished with value: 0.8520979813842532 and parameters: {'n_estimators': 92, 'learning_rate': 0.24924563609040765}. Best is trial 0 with value: 0.8520979813842532.
[I 2026-05-20 23:03:39,835] Trial 4 finished with value: 0.8689736793445423 and para

## Runs Summary

| Run | Optimization | n_estimators | learning_rate | Accuracy (Test) | Accuracy (Train) | F1 (Test) | F1 (Train) | Recall (Test) | Recall (Train) | Fit Time |
|---|---|---:|---:|---:|---:|---:|---:|---:|---:|---:|
| AdaBoost_Baseline | none | 50 | 1.0 | 0.9199 | 0.9213875 | 0.9284757568 | 0.9298939882 | 0.8665 | 0.8689737072 | 6.68s |
| AdaBoost_GridSearch | GridSearchCV | 50 | 1.0 | 0.9199 | 0.9213875 | 0.9284757568 | 0.9298939882 | 0.8665 | 0.8689737072 | 67.06s |
| AdaBoost_Optuna | optuna | 278 | 0.7612632594 | 0.9199 | 0.9213875 | 0.9284757568 | 0.9298939882 | 0.8665 | 0.8689737072 | 247.30s |

### Additional logged parameters
- `random_state = 42` where set in code
- `base_estimator`: Decision stump / default (when applicable)
- Optimized parameters: `n_estimators`, `learning_rate`

## Winner Run Selection

### Policy
A run is only eligible to win if it does **not** show evidence of overfitting or underfitting. Before applying the Recall/F1/fit_time decision rules, we require the **Recall** and **F1** Train→Test gaps (Test − Train) to remain within ±0.5 percentage points (|gap| ≤ 0.005). Runs that fail this generalization check are disqualified.

### Selection Criteria (priority order)
1. **Generalization filter (mandatory):** Recall and F1 gaps within ±0.5 percentage points. Disqualified runs are removed from consideration.
2. **Priority 1 (70%): Highest Recall (Test)** — clinical priority: maximize detection of positive diabetes cases.
3. **Priority 2 (30%): Highest F1-Score (Test)** — used when Recall ties or differs by <0.5% among remaining candidates.
4. **Accuracy is visible but ignored** — shown for reference only; not used in selection.
5. **Tiebreaker: Lowest Fit Time** — if Recall and F1 remain tied.

### Generalization Check
- **AdaBoost_Baseline:** Recall gap = 0.8665 − 0.8689737072 = **−0.25pp** → PASS. F1 gap = 0.9284757568 − 0.9298939882 = **−0.14pp** → PASS.
- **AdaBoost_GridSearch:** Recall gap = 0.8665 − 0.8689737072 = **−0.25pp** → PASS. F1 gap = 0.9284757568 − 0.9298939882 = **−0.14pp** → PASS.
- **AdaBoost_Optuna:** Recall gap = 0.8665 − 0.8689737072 = **−0.25pp** → PASS. F1 gap = 0.9284757568 − 0.9298939882 = **−0.14pp** → PASS.

### Step-by-Step Elimination
**Step 1 — Filter by Highest Test Recall (Priority 1 — 70%)**
- All runs have the same Recall (Test) = **0.8665**.
- Because the gap is zero, we proceed to Priority 2.

**Step 2 — Verify F1 (Priority 2 — 30%)**
- All runs have the same F1 (Test) = **0.9284757568**.
- Because F1 is also tied, we move to the tiebreaker.

**Step 3 — Tiebreaker: Lowest Fit Time**
- `AdaBoost_Baseline`: **6.68s**
- `AdaBoost_GridSearch`: **67.06s**
- `AdaBoost_Optuna`: **247.30s**

### Final Decision
**Winner: AdaBoost_Baseline**

**Justification:** All runs pass the mandatory generalization filter and have identical predictive metrics, so the winner is decided by fit time. `AdaBoost_Baseline` is by far the fastest run and therefore the best operational choice.

### Winner Hyperparameters (AdaBoost_Baseline)

| Parameter | Value |
|---|---|
| **n_estimators** | 50 |
| **learning_rate** | 1.0 |
| **random_state** | 42 |

### Overfitting / Underfitting Diagnosis
- Using the explicit generalization rule (|gap| ≤ 0.5pp on Recall and F1), none of the current runs show disqualifying overfitting or underfitting.
- Since all runs are tied on metric quality, the baseline wins purely on computational efficiency.
